# 🌊 Physics-Informed Neural Networks (PINNs) for Hydrology
# 水文学中的物理信息神经网络 (PINNs)

---

**Welcome, Freshmen!** 欢迎大一新生！

This notebook will teach you about **Physics-Informed Neural Networks (PINNs)** - a cutting-edge technology that combines:
- 🧠 **Artificial Intelligence** (Neural Networks) 
- ⚖️ **Physics Laws** (Water Balance Equation)

**Don't worry if you've never coded before!** We'll explain everything step-by-step.

---

## 📚 Learning Objectives / 学习目标

By the end of this notebook, you will:
1. ✅ Understand what a **Neural Network** is (using simple analogies)
2. ✅ Learn why adding **Physics** makes AI better for science
3. ✅ See how PINNs predict river flow from rainfall
4. ✅ Run your own PINN model and visualize results

---

## 🤔 Step 0: The "Why" - Understanding the Problem

### The Question We Want to Answer:

> **"If I know how much rain fell yesterday, can I predict how much water flows in the river today?"**

This is called the **Rainfall-Runoff Problem** in hydrology.

---

### 🏡 A Simple Analogy: Your Water Tank at Home

Imagine you have a water tank on your roof:

```
Rain ☔ → Tank 🪣 → Faucet 🚰 → Water flows out
```

- **Rain** = Precipitation (P)
- **Evaporation** = Some water disappears in the sun (E)
- **Water in Tank** = Storage (S)
- **Water from Faucet** = River Discharge (Q)

**The Physics Law (Water Balance):**

$$\text{Rain} - \text{Evaporation} - \text{Water Out} = \text{Change in Tank Level}$$

In math notation:

$$P - E - Q = \frac{dS}{dt}$$

**This equation MUST ALWAYS be true!** It's like gravity - you can't break it.

---

## 🧠 Step 1: What is a Neural Network?

### Think of it as a "Smart Function"

**Old Way (Traditional Equation):**
```
Input: Rain = 10 mm
Function: Q = 0.5 × Rain  (we decide this formula)
Output: River Flow = 5 mm
```
**Problem:** What if the relationship isn't so simple? What if it changes with seasons?

---

**New Way (Neural Network):**
```
Input: Rain, Evaporation, Storage
Neural Network: ??? (learns the pattern from data)
Output: River Flow
```

**Benefit:** The network **learns** the complex relationship automatically!

---

### 🏗️ Neural Network Architecture (Like Building Blocks)

```
Input Layer        Hidden Layers           Output Layer
[P, E, S] →  [Neuron] → [Neuron] → [Neuron]  →  [Q, dS]
             [Neuron]   [Neuron]   [Neuron]
             [Neuron]   [Neuron]   [Neuron]
```

- **Input Layer:** Receives the data (rainfall, evaporation, storage)
- **Hidden Layers:** "Thinking" layers that process the information
- **Output Layer:** Produces the prediction (discharge, storage change)

Each connection has a **weight** (like a volume knob) that gets adjusted during training.

---

## ⚖️ Step 2: Why Add Physics?

### The Problem with Pure AI (No Physics)

Imagine training a student **only by memorization** (no understanding):

- Student memorizes: "When it rains 10mm, river flow is 5mm"
- But what if it rains 100mm? The student might predict: "River flow is 2mm" ❌
- **This is nonsense!** More rain should mean more flow!

**Pure data-driven models can learn impossible patterns!**

---

### The PINN Solution: Add Physics as a "Teacher"

We teach the neural network:
1. **Data Teacher:** "Here are 1000 examples of rain and river flow. Learn the pattern."
2. **Physics Teacher:** "Your predictions MUST satisfy: $P - E - Q = dS/dt$. No exceptions!"

**Result:** The network learns realistic patterns that obey physics laws!

---

### 🎯 The Loss Function (How We Train)

During training, we measure two types of errors:

1. **Data Loss (Accuracy):**
   $$L_{data} = \frac{1}{N} \sum (Q_{predicted} - Q_{observed})^2$$
   *"How close are your predictions to real measurements?"*

2. **Physics Loss (Realism):**
   $$L_{physics} = \frac{1}{N} \sum (P - E - Q_{predicted} - dS_{predicted})^2$$
   *"How well do you follow the water balance law?"*

3. **Total Loss (Combined):**
   $$L_{total} = L_{data} + \alpha \times L_{physics}$$
   *$\alpha$ is the "physics weight" - how much we care about physics vs data*

**The network adjusts its weights to minimize this total loss!**

---

## 💻 Step 3: Let's Code! (No Scary Math, I Promise)

### First, Import Libraries

Think of libraries as "toolboxes" - they have pre-made tools we can use.

In [ ]:
# Import libraries (like borrowing tools from a toolbox)
import sys
sys.path.append('..')  # This lets us import from parent directory

import warnings
# Suppress matplotlib font glyph warnings globally
warnings.filterwarnings('ignore', category=UserWarning, message='.*Glyph.*missing from font.*')

import numpy as np  # For math operations on arrays
import matplotlib.pyplot as plt  # For creating plots
import matplotlib as mpl  # For matplotlib configuration
import torch  # PyTorch - the AI library
from datetime import datetime, timedelta
from tqdm import tqdm  # Progress bar (so you can see training progress)

# Import our PINN model
from pinn_model import HydrologyPINN, create_pinn_plots

# Configure matplotlib to handle emojis and Chinese characters properly
# This prevents font warnings when rendering emojis like ☔, 🌡️, 🌊
mpl.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False  # Fix minus sign display

# Try to use Segoe UI Emoji for better emoji support on Windows
try:
    from matplotlib import font_manager
    # Add fallback fonts for emoji rendering
    emoji_fonts = ['Segoe UI Emoji', 'Segoe UI Symbol', 'Apple Color Emoji', 'Noto Color Emoji']
    for font in emoji_fonts:
        if font in [f.name for f in font_manager.fontManager.ttflist]:
            mpl.rcParams['font.sans-serif'].insert(0, font)
            break
except:
    pass  # If font configuration fails, continue with defaults

# Set random seed for reproducibility (so we get the same results every time)
np.random.seed(42)
torch.manual_seed(42)

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"Matplotlib backend: {mpl.get_backend()}")
print(f"Font configuration: {mpl.rcParams['font.sans-serif'][:3]}")
print("📝 Font warnings have been suppressed for cleaner output")


---

## 📊 Step 4: Generate Synthetic Data (Our Practice Dataset)

Before using real data, let's create **synthetic (fake) data** to understand how the model works.

We'll simulate:
- ☔ **Precipitation (P):** Rain events with seasonal patterns
- 🌡️ **Evapotranspiration (E):** Water loss with seasonal patterns
- 🌊 **Discharge (Q):** River flow (this is what we want to predict!)


In [ ]:
# Generate 2 years of daily data
n_days = 730
dates = [datetime(2020, 1, 1) + timedelta(days=i) for i in range(n_days)]
t = np.arange(n_days)

print("🌧️ Generating synthetic hydrology data...")

# 1. Precipitation (with realistic seasonal pattern)
# More rain in winter (when sin is negative)
seasonal_factor = 1.5 + 0.8 * np.sin(2 * np.pi * t / 365 + np.pi)
P_base = np.random.gamma(1.5, 4, n_days) * seasonal_factor

# Add dry days (60-80% of days have no rain, varies by season)
dry_prob = 0.6 + 0.2 * np.sin(2 * np.pi * t / 365)
P = np.where(np.random.rand(n_days) < dry_prob, 0, P_base)

# Add a few extreme rainfall events
extreme_events = np.random.choice(n_days, size=5, replace=False)
P[extreme_events] += np.random.gamma(5, 10, 5)

# 2. Evapotranspiration (higher in summer)
E_mean = 4.0
E_amplitude = 2.5
E = E_mean + E_amplitude * np.sin(2 * np.pi * t / 365) + np.random.normal(0, 0.3, n_days)
E = np.maximum(E, 0.5)  # ET can't be negative!

# 3. Generate "true" discharge using a simple reservoir model
Q_true = np.zeros(n_days)
S = np.zeros(n_days)
S[0] = 100.0  # Initial storage in mm

for i in range(n_days):
    # Water balance: what goes in minus what goes out
    inflow = max(0, P[i] - E[i])
    outflow = 0.1 * S[i]  # Linear reservoir: discharge proportional to storage
    
    Q_true[i] = outflow
    
    if i < n_days - 1:
        S[i + 1] = S[i] + inflow - outflow
        S[i + 1] = max(0, S[i + 1])  # Storage can't be negative!

# Add realistic measurement noise to create "observed" data
Q_obs = Q_true + np.random.normal(0, 0.1 * np.std(Q_true), n_days)
Q_obs = np.maximum(Q_obs, 0)  # Discharge can't be negative!

print("✅ Data generation complete!")
print(f"\n📈 Data Statistics:")
print(f"  Period: {n_days} days ({n_days/365:.1f} years)")
print(f"  Mean Precipitation: {np.mean(P):.2f} mm/day")
print(f"  Mean Evapotranspiration: {np.mean(E):.2f} mm/day")
print(f"  Mean Discharge: {np.mean(Q_obs):.2f} mm/day")
print(f"  Max Precipitation Event: {np.max(P):.2f} mm/day")
print(f"  Rainy Days: {np.sum(P > 0.1)} ({np.sum(P > 0.1)/n_days*100:.1f}%)")

---

## 🎨 Step 5: Visualize the Data (What Does It Look Like?)

**Before training any model, ALWAYS look at your data first!**

Let's create plots to understand:
- When does it rain?
- How does river flow respond to rainfall?
- Are there seasonal patterns?

In [ ]:
# Create visualization
from matplotlib.font_manager import FontProperties

# Set up font properties for Chinese characters
cn_font = FontProperties(family=['Segoe UI Emoji', 'Microsoft YaHei', 'SimHei'])

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
fig.suptitle('🌊 Our Hydrology Dataset / 我们的水文数据集', fontsize=16, fontweight='bold', fontproperties=cn_font)

# Plot 1: Precipitation (upside down, like it's falling!)
axes[0].bar(dates, P, color='steelblue', alpha=0.7, width=1)
axes[0].set_ylabel('Precipitation / 降水\n(mm/day)', fontweight='bold', fontsize=12, fontproperties=cn_font)
axes[0].invert_yaxis()  # Flip it upside down
axes[0].set_ylim(max(P) * 1.1, 0)
axes[0].grid(True, alpha=0.3)
axes[0].set_title('☔ Rainfall Events / 降雨事件', fontsize=12, fontproperties=cn_font)

# Plot 2: Evapotranspiration
axes[1].plot(dates, E, color='orange', linewidth=1.5, label='Evapotranspiration')
axes[1].fill_between(dates, E, alpha=0.3, color='orange')
axes[1].set_ylabel('Evapotranspiration / 蒸发\n(mm/day)', fontweight='bold', fontsize=12, fontproperties=cn_font)
axes[1].grid(True, alpha=0.3)
axes[1].set_title('🌡️ Water Loss to Atmosphere / 大气蒸发', fontsize=12, fontproperties=cn_font)
axes[1].legend(fontsize=10)

# Plot 3: Observed Discharge (what we want to predict!)
axes[2].plot(dates, Q_obs, color='darkblue', linewidth=2, label='Observed Discharge')
axes[2].fill_between(dates, Q_obs, alpha=0.3, color='darkblue')
axes[2].set_ylabel('Discharge / 径流\n(mm/day)', fontweight='bold', fontsize=12, fontproperties=cn_font)
axes[2].set_xlabel('Date / 日期', fontweight='bold', fontsize=12, fontproperties=cn_font)
axes[2].grid(True, alpha=0.3)
axes[2].set_title('🌊 River Flow (Target for Prediction) / 河流流量（预测目标）', fontsize=12, fontproperties=cn_font)
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\n🤔 Questions to Think About:")
print("  1. Do you see the river flow increase after big rainfall events?")
print("  2. Is there a time delay between rain and river response?")
print("  3. What happens during dry periods (no rain)?")


---

## 🏗️ Step 6: Build the PINN Model

Now let's create our Physics-Informed Neural Network!

### Model Configuration (These are called "Hyperparameters")

- **hidden_layers = [64, 32, 16]**: Three "thinking" layers with 64, 32, and 16 neurons
  - *More neurons = can learn more complex patterns, but slower training*
  
- **activation = 'tanh'**: The "activation function" (how neurons decide to fire)
  - *tanh is good for smooth, continuous data like river flow*
  
- **physics_weight = 0.1**: How much we care about physics vs. data
  - *0.0 = pure data-driven (ignore physics)*
  - *1.0 = equal weight to physics and data*
  
- **learning_rate = 0.001**: How fast we learn
  - *Too high = unstable, too low = takes forever*

In [ ]:
# Create the PINN model
print("🏗️ Building the PINN model...")

model = HydrologyPINN(
    hidden_layers=[64, 32, 16],    # Neural network architecture
    activation='tanh',              # Activation function
    physics_weight=0.1,             # Weight for physics loss
    learning_rate=0.001             # Learning rate
)

print("✅ Model created successfully!")
print(f"\n🧠 Model Architecture:")
print(f"  Input Layer: 3 features [P, E, S_prev]")
print(f"  Hidden Layers: {model.hidden_layers}")
print(f"  Output Layer: 2 predictions [Q, dS]")
print(f"  Total Parameters: {sum(p.numel() for p in model.parameters())}")
print(f"\n⚙️ Training Settings:")
print(f"  Physics Weight: {model.physics_weight}")
print(f"  Learning Rate: {model.learning_rate}")

---

## 🎓 Step 7: Train the Model (This is Where the Magic Happens!)

**Training = Learning from data**

The model will:
1. Look at the data (P, E, Q)
2. Make predictions
3. Calculate errors (data loss + physics loss)
4. Adjust its weights to reduce errors
5. Repeat steps 2-4 many times (epochs)

**Watch the progress bar!** Each epoch takes a few seconds.

### What to Expect:
- **Loss should decrease** over time (the model is learning!)
- Training on **80% of data**, validating on **20%** (to check if it really learned or just memorized)

In [ ]:
# Train the model
print("🎓 Training the PINN model...")
print("This will take about 1-2 minutes. Watch the loss decrease!\n")

history = model.fit(
    P=P,
    E=E,
    Q_obs=Q_obs,
    epochs=200,              # Number of complete passes through the data
    batch_size=32,           # Process 32 samples at a time
    verbose=True,            # Print progress
    validation_split=0.2     # Use 20% of data for validation
)

print("\n🎉 Training complete!")

---

## 📈 Step 8: Visualize Training Progress (Did It Learn?)

Let's plot the **loss curves** to see how the model improved over time.

**What to look for:**
- ✅ **Decreasing loss** = Model is learning!
- ✅ **Smooth curve** = Stable training
- ❌ **Flat line** = Not learning (might need to adjust settings)
- ❌ **Zigzag pattern** = Unstable (learning rate might be too high)

In [ ]:
# Plot training history
from matplotlib.font_manager import FontProperties

# Set up font properties for Chinese characters
cn_font = FontProperties(family=['Segoe UI Emoji', 'Microsoft YaHei', 'SimHei'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('📊 PINN Training History / PINN训练历史', fontsize=16, fontweight='bold', fontproperties=cn_font)

epochs_range = range(1, len(history['total_loss']) + 1)

# Plot 1: Total Loss
axes[0].plot(epochs_range, history['total_loss'], 'b-', linewidth=2, label='Total Loss')
axes[0].set_xlabel('Epoch / 训练轮次', fontweight='bold', fontsize=12, fontproperties=cn_font)
axes[0].set_ylabel('Loss / 损失', fontweight='bold', fontsize=12, fontproperties=cn_font)
axes[0].set_title('📉 Total Loss (Should Decrease!) / 总损失（应该下降！）', fontsize=12, fontproperties=cn_font)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)

# Plot 2: Data Loss vs Physics Loss
axes[1].plot(epochs_range, history['data_loss'], 'r-', linewidth=2, label='Data Loss / 数据损失')
axes[1].plot(epochs_range, history['physics_loss'], 'g-', linewidth=2, label='Physics Loss / 物理损失')
axes[1].set_xlabel('Epoch / 训练轮次', fontweight='bold', fontsize=12, fontproperties=cn_font)
axes[1].set_ylabel('Loss / 损失', fontweight='bold', fontsize=12, fontproperties=cn_font)
axes[1].set_title('⚖️ Data Loss vs Physics Loss / 数据损失 vs 物理损失', fontsize=12, fontproperties=cn_font)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=10, prop=cn_font)

plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print(f"  Final Total Loss: {history['total_loss'][-1]:.6f}")
print(f"  Final Data Loss: {history['data_loss'][-1]:.6f}")
print(f"  Final Physics Loss: {history['physics_loss'][-1]:.6f}")
print("\n  Both losses decreased → The model learned to fit data AND obey physics! ✅")


---

## 🔮 Step 9: Make Predictions (Test the Model!)

Now let's use our trained model to predict river discharge!

The model will:
1. Take inputs: P (rain), E (evaporation)
2. Predict: Q (river flow)
3. Update: S (storage) for the next timestep

In [ ]:
# Make predictions
print("🔮 Making predictions...")

Q_pred = model.predict(P, E)

print("✅ Predictions complete!")
print(f"\n📊 Prediction Statistics:")
print(f"  Mean Predicted Discharge: {np.mean(Q_pred):.2f} mm/day")
print(f"  Max Predicted Discharge: {np.max(Q_pred):.2f} mm/day")
print(f"  Min Predicted Discharge: {np.min(Q_pred):.2f} mm/day")

---

## 📊 Step 10: Evaluate Performance (How Good Is It?)

We'll use standard metrics to evaluate our model:

### 📏 Performance Metrics Explained:

1. **NSE (Nash-Sutcliffe Efficiency)**
   - Range: -∞ to 1.0
   - **1.0 = Perfect!** Your prediction matches reality exactly
   - **0.0 = As good as just using the average** (not learning anything)
   - **< 0 = Worse than average** (something went wrong!)
   - **> 0.5 = Good** for hydrology
   - **> 0.7 = Excellent!**

2. **R² (R-Squared)**
   - Range: 0 to 1.0
   - Measures correlation (how well predictions follow the trend)
   - **> 0.7 = Strong correlation**

3. **RMSE (Root Mean Square Error)**
   - Average error in mm/day
   - **Lower is better**
   - Compare to mean discharge to see if it's reasonable

4. **PBIAS (Percent Bias)**
   - **0% = Perfect** (no systematic over/under-prediction)
   - **Positive = Underestimating** (predicting too low)
   - **Negative = Overestimating** (predicting too high)
   - **±10% = Good** for discharge

In [ ]:
# Calculate performance metrics
metrics = model.calculate_metrics(Q_obs, Q_pred)

print("\n" + "="*70)
print("🎯 Model Performance Metrics / 模型性能指标")
print("="*70)
print(f"\n  NSE (Nash-Sutcliffe Efficiency): {metrics['NSE']:.3f}")
if metrics['NSE'] > 0.7:
    print("    → Excellent! 优秀！")
elif metrics['NSE'] > 0.5:
    print("    → Good! 良好！")
elif metrics['NSE'] > 0:
    print("    → Acceptable 可接受")
else:
    print("    → Needs improvement 需要改进")

print(f"\n  R² (Coefficient of Determination): {metrics['R2']:.3f}")
if metrics['R2'] > 0.7:
    print("    → Strong correlation! 强相关！")
elif metrics['R2'] > 0.5:
    print("    → Moderate correlation 中等相关")
else:
    print("    → Weak correlation 弱相关")

print(f"\n  RMSE (Root Mean Square Error): {metrics['RMSE']:.3f} mm/day")
print(f"    → Average error is {metrics['RMSE']:.3f} mm/day")
print(f"    → This is {metrics['RMSE']/np.mean(Q_obs)*100:.1f}% of mean discharge")

print(f"\n  PBIAS (Percent Bias): {metrics['PBIAS']:.2f}%")
if abs(metrics['PBIAS']) < 10:
    print("    → Low bias! 低偏差！")
elif abs(metrics['PBIAS']) < 25:
    print("    → Moderate bias 中等偏差")
else:
    print("    → High bias 高偏差")

if metrics['PBIAS'] > 0:
    print("    → Model tends to underestimate (预测偏低)")
elif metrics['PBIAS'] < 0:
    print("    → Model tends to overestimate (预测偏高)")
else:
    print("    → Perfect balance! (完美平衡！)")

print("\n" + "="*70)

---

## 🎨 Step 11: Visualize Predictions (The Moment of Truth!)

Let's create beautiful plots to see how well our model works!

In [ ]:
# Create comprehensive visualization
from matplotlib.font_manager import FontProperties

# Set up font properties for Chinese characters
cn_font = FontProperties(family=['Segoe UI Emoji', 'Microsoft YaHei', 'SimHei'])

fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True)
fig.suptitle('🌊 PINN Model Results / PINN模型结果', fontsize=18, fontweight='bold', fontproperties=cn_font)

# Plot 1: Precipitation
axes[0].bar(dates, P, color='steelblue', alpha=0.7, width=1)
axes[0].set_ylabel('Precipitation\n(mm/day)', fontweight='bold', fontsize=11)
axes[0].invert_yaxis()
axes[0].set_ylim(max(P) * 1.1, 0)
axes[0].grid(True, alpha=0.3)
axes[0].set_title('☔ Input: Rainfall', fontsize=12, fontproperties=cn_font)

# Plot 2: Evapotranspiration
axes[1].plot(dates, E, color='orange', linewidth=1.5)
axes[1].fill_between(dates, E, alpha=0.3, color='orange')
axes[1].set_ylabel('Evapotranspiration\n(mm/day)', fontweight='bold', fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_title('🌡️ Input: Evaporation', fontsize=12, fontproperties=cn_font)

# Plot 3: Discharge Comparison (The main plot!)
axes[2].plot(dates, Q_obs, 'k-', linewidth=2, label='Observed (Truth)', alpha=0.8)
axes[2].plot(dates, Q_pred, 'r--', linewidth=2, label='PINN Prediction', alpha=0.8)
axes[2].fill_between(dates, Q_obs, alpha=0.2, color='black', label='Observed range')
axes[2].fill_between(dates, Q_pred, alpha=0.2, color='red', label='Predicted range')
axes[2].set_ylabel('Discharge\n(mm/day)', fontweight='bold', fontsize=11)
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc='upper right', fontsize=10)
axes[2].set_title(f'🌊 Output: River Flow (NSE={metrics["NSE"]:.3f})', fontsize=12, fontproperties=cn_font)

# Plot 4: Residuals (Errors)
residuals = Q_obs - Q_pred
axes[3].plot(dates, residuals, color='purple', linewidth=1.5, label='Error (Obs - Pred)')
axes[3].axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)
axes[3].fill_between(dates, residuals, alpha=0.3, color='purple')
axes[3].set_ylabel('Residuals\n(mm/day)', fontweight='bold', fontsize=11)
axes[3].set_xlabel('Date', fontweight='bold', fontsize=12)
axes[3].grid(True, alpha=0.3)
axes[3].legend(loc='upper right', fontsize=10)
axes[3].set_title('📊 Prediction Errors (Should be random around zero)', fontsize=12, fontproperties=cn_font)

plt.tight_layout()
plt.show()

print("\n🔍 What to look for in the plots:")
print("  1. Does the PINN prediction (red) follow the observed (black)?")
print("  2. Are peak flows captured well?")
print("  3. Are residuals (errors) randomly scattered around zero?")
print("  4. Is there a time lag between rain and river response?")


---

## 🎯 Step 12: Scatter Plot (Observed vs Predicted)

This plot shows:
- **X-axis:** What actually happened (observed discharge)
- **Y-axis:** What our model predicted
- **Red line:** Perfect 1:1 line (if all points were on this line, predictions would be perfect!)

**Good signs:**
- Points clustered near the 1:1 line
- No systematic pattern (not all above or below the line)

In [ ]:
# Scatter plot
from matplotlib.font_manager import FontProperties

# Set up font properties for Chinese characters
cn_font = FontProperties(family=['Segoe UI Emoji', 'Microsoft YaHei', 'SimHei'])

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Scatter plot with color gradient based on precipitation
scatter = ax.scatter(Q_obs, Q_pred, c=P, cmap='Blues', alpha=0.6, s=50, 
                     edgecolors='black', linewidth=0.5)

# 1:1 line (perfect prediction)
max_val = max(np.max(Q_obs), np.max(Q_pred))
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=3, label='Perfect Prediction (1:1 line)', alpha=0.8)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Precipitation (mm/day)', fontweight='bold', fontsize=11)

# Add metrics box
metrics_text = f"Performance Metrics:\n"
metrics_text += f"NSE: {metrics['NSE']:.3f}\n"
metrics_text += f"R²: {metrics['R2']:.3f}\n"
metrics_text += f"RMSE: {metrics['RMSE']:.3f} mm/day\n"
metrics_text += f"PBIAS: {metrics['PBIAS']:.2f}%"

ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes,
        bbox=dict(boxstyle="round", facecolor='white', alpha=0.9, edgecolor='black', linewidth=2),
        verticalalignment='top', fontweight='bold', fontsize=12)

ax.set_xlabel('Observed Discharge (mm/day)', fontweight='bold', fontsize=13)
ax.set_ylabel('Predicted Discharge (mm/day)', fontweight='bold', fontsize=13)
ax.set_title('🎯 PINN Performance: Observed vs Predicted\nPINN性能：观测 vs 预测', 
             fontweight='bold', fontsize=15, fontproperties=cn_font)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("\n💡 Interpretation Guide:")
print("  Points near the red line → Good predictions! ✅")
print("  Points above the line → Model overestimates (predicts too high)")
print("  Points below the line → Model underestimates (predicts too low)")
print("  Color shows precipitation: Darker blue = more rain")


---

## 🧪 Step 13: Experiment Time! (Try Changing Things)

Now that you understand how PINNs work, let's experiment!

### 🔬 Experiment Ideas:

1. **Change Physics Weight:**
   - Try `physics_weight = 0.0` (pure data-driven, no physics)
   - Try `physics_weight = 1.0` (strong physics constraint)
   - **Question:** Which gives better results?

2. **Change Network Architecture:**
   - Try `hidden_layers = [32]` (smaller, faster)
   - Try `hidden_layers = [128, 64, 32, 16]` (bigger, more complex)
   - **Question:** Does bigger always mean better?

3. **Change Training Duration:**
   - Try `epochs = 50` (faster, less learning)
   - Try `epochs = 500` (slower, more learning)
   - **Question:** When does it stop improving?

**To experiment:** Go back to Step 6, change the parameters, and re-run cells 6-12!

In [ ]:
# Comparison experiment: Pure AI vs PINN
print("🧪 Experiment: Comparing Pure AI vs PINN")
print("="*70)

# Model 1: Pure Data-Driven (no physics)
print("\n1️⃣ Training Pure AI Model (physics_weight=0.0)...")
model_pure = HydrologyPINN(
    hidden_layers=[64, 32, 16],
    physics_weight=0.0,  # NO PHYSICS!
    learning_rate=0.001
)
history_pure = model_pure.fit(P, E, Q_obs, epochs=200, batch_size=32, verbose=False)
Q_pred_pure = model_pure.predict(P, E)
metrics_pure = model_pure.calculate_metrics(Q_obs, Q_pred_pure)

# Model 2: PINN (with physics)
print("\n2️⃣ Training PINN Model (physics_weight=0.1)...")
model_pinn = HydrologyPINN(
    hidden_layers=[64, 32, 16],
    physics_weight=0.1,  # WITH PHYSICS!
    learning_rate=0.001
)
history_pinn = model_pinn.fit(P, E, Q_obs, epochs=200, batch_size=32, verbose=False)
Q_pred_pinn = model_pinn.predict(P, E)
metrics_pinn = model_pinn.calculate_metrics(Q_obs, Q_pred_pinn)

# Compare results
print("\n" + "="*70)
print("📊 Comparison Results")
print("="*70)
print(f"\n{'Metric':<20} {'Pure AI':<15} {'PINN':<15} {'Winner'}")
print("-"*70)
print(f"{'NSE':<20} {metrics_pure['NSE']:<15.3f} {metrics_pinn['NSE']:<15.3f} "
      f"{'PINN ✅' if metrics_pinn['NSE'] > metrics_pure['NSE'] else 'Pure AI ✅'}")
print(f"{'R²':<20} {metrics_pure['R2']:<15.3f} {metrics_pinn['R2']:<15.3f} "
      f"{'PINN ✅' if metrics_pinn['R2'] > metrics_pure['R2'] else 'Pure AI ✅'}")
print(f"{'RMSE':<20} {metrics_pure['RMSE']:<15.3f} {metrics_pinn['RMSE']:<15.3f} "
      f"{'PINN ✅' if metrics_pinn['RMSE'] < metrics_pure['RMSE'] else 'Pure AI ✅'}")
print(f"{'|PBIAS|':<20} {abs(metrics_pure['PBIAS']):<15.2f} {abs(metrics_pinn['PBIAS']):<15.2f} "
      f"{'PINN ✅' if abs(metrics_pinn['PBIAS']) < abs(metrics_pure['PBIAS']) else 'Pure AI ✅'}")
print("\n💡 Conclusion:")
if metrics_pinn['NSE'] > metrics_pure['NSE']:
    print("  PINN (with physics) performs better! Physics helps! 🎉")
else:
    print("  Interesting! Pure AI performed better in this case.")
    print("  This might happen when: (1) data is very clean, (2) physics weight is too high")

---

## 🎓 Step 14: Summary and Key Takeaways

### What You Learned Today:

1. ✅ **Neural Networks** are like smart functions that learn patterns from data

2. ✅ **PINNs** combine data learning with physics laws to make predictions more realistic

3. ✅ **Water Balance Equation** ($P - E - Q = dS/dt$) is a fundamental physics law that must be satisfied

4. ✅ **Loss Function** has two parts:
   - **Data Loss:** How close predictions are to observations
   - **Physics Loss:** How well predictions satisfy physics laws

5. ✅ **Performance Metrics** help us evaluate model quality:
   - NSE > 0.5 = Good
   - RMSE should be small compared to mean
   - PBIAS near 0% is ideal

---

### 🌟 Why PINNs are Revolutionary:

Traditional Models (like GR4J, HBV):
- ✅ Based on physics
- ✅ Interpretable
- ❌ Hard to calibrate
- ❌ Fixed structure

Pure AI Models (like LSTM):
- ✅ Flexible
- ✅ Can learn complex patterns
- ❌ Black box (hard to interpret)
- ❌ Can predict nonsense

**PINNs (Best of Both Worlds!):**
- ✅ Flexible like AI
- ✅ Constrained by physics
- ✅ More interpretable than pure AI
- ✅ Makes physically realistic predictions

---

### 🚀 Next Steps:

1. **Try different hyperparameters** (physics weight, network size, learning rate)
2. **Apply to real data** (download real river flow data from your local watershed)
3. **Compare with traditional models** (GR4J, HBV from other notebooks)
4. **Explore advanced topics:**
   - Multi-variable PINNs (include temperature, soil moisture)
   - Spatially-distributed PINNs
   - Transfer learning (train on one watershed, apply to another)

---

### 📚 Further Reading:

**Papers:**
- Raissi et al. (2019). "Physics-informed neural networks" - Original PINN paper
- Feng et al. (2023). "Differentiable, learnable, regionalized process-based models" - PINNs for hydrology

**Resources:**
- [PyTorch Tutorials](https://pytorch.org/tutorials/)
- [Deep Learning for Physical Scientists](https://arxiv.org/abs/1903.10563)

---

## 🎉 Congratulations!

You've completed the PINN tutorial! You now understand one of the most exciting frontiers in scientific machine learning.

**恭喜你！** 你已经完成了PINN教程！你现在理解了科学机器学习中最激动人心的前沿领域之一。

---

## 🤔 Reflection Questions (课后思考题)

1. **Why do we normalize the input data?**
   - *Hint: Neural networks work best when all inputs are on similar scales*

2. **What happens if we set physics_weight = 0?**
   - *Try it and compare the results!*

3. **Can PINNs predict future climate change impacts?**
   - *Think about: What would change? Would the physics laws still hold?*

4. **Why might a PINN perform worse than a traditional model?**
   - *Consider: Data quality, amount of training data, model complexity*

5. **How could you improve this PINN model?**
   - *Ideas: Add more input features, use longer training, tune hyperparameters*

---

## 📝 Notes Space (Your Personal Notes)

Use this space to write your observations, questions, and insights!

---

**My observations:**



**Questions I have:**



**Interesting findings:**



---